In [6]:
import torch
import os
import random
import shutil
import numpy as np
import traceback
from denoising_diffusion_pytorch import Unet, GaussianDiffusion, Trainer

In [ ]:
def configurar_semente(seed=42, modo_rapido=True):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    if modo_rapido:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
    else:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

configurar_semente(42, modo_rapido=True)

In [2]:
base_dataset = r"C:\Analise_Esporos_2026\Dataset_Recortado_4_classes"

base_splits = r"C:\Analise_Esporos_2026\Difusao_ModeloAnterior\Dataset_Splits"
os.makedirs(base_splits, exist_ok=True)

classes = [
    "mildio",
    "oidio",
    "puccinias",
    "phakopsora"
]

nums = [100, 300, 500, 700, 1000, 1300]

# Garantir que pega apenas imagens
extensions = (".png", ".jpg", ".jpeg")

SEED = 42

for classe in classes:

    dataset_folder = os.path.join(
        base_dataset,
        classe
    )

    split_folder = os.path.join(
        base_splits,
        classe
    )

    os.makedirs(split_folder, exist_ok=True)

    all_images = [
        filename
        for filename in os.listdir(dataset_folder)
        if filename.lower().endswith(extensions)
    ]

    all_images.sort()

    rng = random.Random(SEED)
    rng.shuffle(all_images)

    print(f"\nClass: {classe}")
    print(f"Available images: {len(all_images)}")

    max_num = max(nums)

    if len(all_images) < max_num:
        print(
            f"WARNING: {classe} has only {len(all_images)} images, "
            f"but {max_num} were requested."
        )

    for num in nums:

        # Ele pega as primeiras imagens até o limitador (num)
        selected = all_images[:min(num, len(all_images))]

        output_file = os.path.join(
            split_folder,
            f"{num}.txt"
        )

        with open(
            output_file,
            "w",
            encoding="utf-8"
        ) as f:

            for filename in selected:
                f.write(filename + "\n")

        print(
            f"  {num:4d} images -> {len(selected):4d} saved -> "
            f"{output_file}"
        )


Class: mildio
Available images: 1438
   100 images ->  100 saved -> C:\Analise_Esporos_2026\Difusao_ModeloAnterior\Dataset_Splits\mildio\100.txt
   300 images ->  300 saved -> C:\Analise_Esporos_2026\Difusao_ModeloAnterior\Dataset_Splits\mildio\300.txt
   500 images ->  500 saved -> C:\Analise_Esporos_2026\Difusao_ModeloAnterior\Dataset_Splits\mildio\500.txt
   700 images ->  700 saved -> C:\Analise_Esporos_2026\Difusao_ModeloAnterior\Dataset_Splits\mildio\700.txt
  1000 images -> 1000 saved -> C:\Analise_Esporos_2026\Difusao_ModeloAnterior\Dataset_Splits\mildio\1000.txt
  1300 images -> 1300 saved -> C:\Analise_Esporos_2026\Difusao_ModeloAnterior\Dataset_Splits\mildio\1300.txt

Class: oidio
Available images: 3774
   100 images ->  100 saved -> C:\Analise_Esporos_2026\Difusao_ModeloAnterior\Dataset_Splits\oidio\100.txt
   300 images ->  300 saved -> C:\Analise_Esporos_2026\Difusao_ModeloAnterior\Dataset_Splits\oidio\300.txt
   500 images ->  500 saved -> C:\Analise_Esporos_2026\Difusa

In [ ]:
classe = "mildio"
# "mildio"
# "oidio"
# "puccinias"
# "phakopsora"

nums = [100, 300, 500, 700, 1000, 1300]

base_dataset = r"C:\Analise_Esporos_2026\Dataset_Recortado_4_classes"

base_splits = r"C:\Analise_Esporos_2026\Difusao_ModeloAnterior\Dataset_Splits"

base_modelos = r"C:\Analise_Esporos_2026\Difusao_ModeloAnterior\Modelos_Experimentos"

base_imgs = os.path.join(
    base_modelos,
    "imgs"
)

for num in nums:

    manifest_path = os.path.join(
        base_splits,
        classe,
        f"{num}.txt"
    )

    source_folder = os.path.join(
        base_dataset,
        classe
    )

    train_folder = os.path.join(
        base_imgs,
        classe,
        str(num)
    )

    results_folder = os.path.join(
        base_modelos,
        f"modelos_{num}_imagens",
        f"results_{classe}"
    )

    os.makedirs(train_folder, exist_ok=True)
    os.makedirs(results_folder, exist_ok=True)

    with open(
        manifest_path,
        "r",
        encoding="utf-8"
    ) as f:

        filenames = [
            line.strip()
            for line in f
            if line.strip()
        ]

    print(f"Manifest contains {len(filenames)} images.")

    for filename in filenames:

        src = os.path.join(
            source_folder,
            filename
        )

        if not os.path.isfile(src):
            raise FileNotFoundError(
                f"Missing image:\n{src}"
            )

    for filename in filenames:

        src = os.path.join(
            source_folder,
            filename
        )

        dst = os.path.join(
            train_folder,
            filename
        )

        if not os.path.exists(dst):
            shutil.copy2(src, dst)

    model = Unet(
        dim=64,
        dim_mults=(1, 2, 4, 8),
        flash_attn=False
    )

    diffusion = GaussianDiffusion(
        model,
        image_size=128,
        timesteps=1000
    )

    trainer = Trainer(
        diffusion,
        train_folder,
        train_batch_size=16,
        train_lr=8e-5,
        train_num_steps=30000,
        gradient_accumulate_every=2,
        ema_decay=0.995,
        amp=True,
        calculate_fid=False,
        save_and_sample_every=5000,
        results_folder=results_folder
    )

    print("Starting training...")
    
    
    try:
        trainer.train()
        
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"\n[ERROR] GPU Out of Memory on dataset size {num}!")
            print("Skipping to next size. Consider reducing batch_size.")
            if torch.cuda.is_available():
                torch.cuda.empty_cache() # Limpa a memória residual
        else:
            print(f"\n[ERROR] PyTorch RuntimeError on dataset size {num}: {e}")
            traceback.print_exc()
            
    except Exception as e:
        print(f"\n[CRITICAL ERROR] Unexpected failure on dataset size {num}: {e}")
        traceback.print_exc()

    print(f"Finished {classe} - {num} images.")

Manifest contains 100 images.
Starting training...


  0%|          | 0/30000 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

In [4]:
import torch

print(torch.cuda.get_device_name(0))
print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")
print(f"Total:     {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")

NVIDIA GeForce RTX 3050
Allocated: 3.32 GB
Reserved:  7.54 GB
Total:     6.00 GB


In [5]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   3396 MiB |   7430 MiB | 214031 GiB | 214027 GiB |
|       from large pool |   3271 MiB |   7265 MiB | 212960 GiB | 212956 GiB |
|       from small pool |    125 MiB |    188 MiB |   1071 GiB |   1070 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   3396 MiB |   7430 MiB | 214031 GiB | 214027 GiB |
|       from large pool |   3271 MiB |   7265 MiB | 212960 GiB |

In [ ]:
import torch
import torchvision
import denoising_diffusion_pytorch
import accelerate
import einops

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("DDPM:", denoising_diffusion_pytorch.__version__)
print("Accelerate:", accelerate.__version__)
print("Einops:", einops.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
nums = [100, 300, 500, 700, 1000, 1300]
classe = 'mildio'
# 'mildio', 'oidio', 'puccinias', 'phakopsora'

base_modelos = r'C:\Analise_Esporos_2026\Difusao_ModeloAnterior\Modelos_Experimentos'
os.makedirs(base_modelos, exist_ok=True)
base_dataset = r'C:\Analise_Esporos_2026\Dataset_Recortado_4_classes'
base_imgs = os.path.join(base_modelos, "imgs")

for num in nums:
    pasta = os.path.join(base_modelos, f"modelos_{num}_imagens")
    os.makedirs(pasta, exist_ok=True)

    # Garantir que exista a pasta imgs/{classe}
    pasta_destino = os.path.join(base_imgs, classe)
    os.makedirs(pasta_destino, exist_ok=True)
    
    pasta_dataset_curado = os.path.join(base_dataset, classe)
    arquivos_atuais = [arq for arq in os.listdir(pasta_destino) if arq.lower().endswith((".png", ".jpg", ".jpeg"))]
    arquivos_dataset = [arq for arq in os.listdir(pasta_dataset_curado) if arq.lower().endswith((".png", ".jpg", ".jpeg"))]
    atual = len(arquivos_atuais)
    mudar = num - atual

    if mudar > 0:
        # Pegar candidatos que ainda não foram copiados
        candidatos = list(set(arquivos_dataset) - set(arquivos_atuais))
        escolhidos = random.sample(candidatos, min(mudar, len(candidatos)))

        for arquivo in escolhidos:
            src = os.path.join(pasta_dataset_curado, arquivo)
            dst = os.path.join(pasta_destino, arquivo)
            shutil.copy(src, dst)

    # Agora treina o modelo
    model = Unet(
        dim=64,
        dim_mults=(1, 2, 4, 8),
        flash_attn=False
    )

    diffusion = GaussianDiffusion(
        model,
        image_size=128,
        timesteps=1000
    )

    trainer = Trainer(
        diffusion,
        os.path.join(base_imgs, classe),
        train_batch_size=16,
        train_lr=8e-5,
        train_num_steps=30000,
        gradient_accumulate_every=2,
        ema_decay=0.995,
        amp=True,
        calculate_fid=False,
        save_and_sample_every=5000,
        results_folder=os.path.join(pasta, f"results_{classe}")
    )

    # Inicia o treino
    trainer.train()

  0%|          | 0/30000 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

... (11 outputs left out)

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("Compute capability:", torch.cuda.get_device_capability(0))

print("Flash SDP enabled:", torch.backends.cuda.flash_sdp_enabled())
print("Memory-efficient SDP enabled:", torch.backends.cuda.mem_efficient_sdp_enabled())
print("Math SDP enabled:", torch.backends.cuda.math_sdp_enabled())

PyTorch: 2.13.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050
CUDA version: 12.6
Compute capability: (8, 6)
Flash SDP enabled: True
Memory-efficient SDP enabled: True
Math SDP enabled: True


In [5]:
params = torch.backends.cuda.SDPAParams(
    q,
    k,
    v,
    None,
    0.0,
    False,  # is_causal
    False   # enable_gqa
)

print(
    "Flash usable:",
    torch.backends.cuda.can_use_flash_attention(
        params,
        debug=True
    )
)

print(
    "Efficient usable:",
    torch.backends.cuda.can_use_efficient_attention(
        params,
        debug=True
    )
)

print(
    "Math usable:",
    torch.backends.cuda.can_use_math_attention(
        params,
        debug=True
    )
)

Flash usable: False
Efficient usable: True


AttributeError: module 'torch.backends.cuda' has no attribute 'can_use_math_attention'

In [6]:
import torch
import torch.nn.functional as F

q = torch.randn(16, 4, 256, 64, device="cuda", dtype=torch.float16)
k = torch.randn(16, 4, 256, 64, device="cuda", dtype=torch.float16)
v = torch.randn(16, 4, 256, 64, device="cuda", dtype=torch.float16)

with torch.nn.attention.sdpa_kernel(
    torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION
):
    out = F.scaled_dot_product_attention(q, k, v)

print("Efficient attention worked:", out.shape)

Efficient attention worked: torch.Size([16, 4, 256, 64])
